# exp091 Analysis Inference — Labeled SS + train_audio + train_sc

**Purpose**: Compute predictions on training data to enable LOCAL Streamlit analysis.

## Outputs (saved to /kaggle/working/, downloadable)

| File | Shape | 内容 |
|---|---|---|
| `truth_ss.npz` | (708, 234) | labeled SS 真実ラベル |
| `perch_emb_ss.npz` | (708, 1536) | Perch v2 embeddings on labeled SS |
| `perch_score_ss.npz` | (708, 234) | Perch v2 logits (mapped to BC2026 labels) |
| `exp037_oof_ss.npz` | (708, 234) | exp037 v313 stream OOF (full pipeline) |
| `meta_ss.parquet` | 708 rows | filename, site, hour, window_idx |
| `perch_score_train_audio.npz` | (35549, 234) | Perch v2 on center-5s of focal |
| `meta_train_audio.parquet` | 35549 rows | filename, primary_label, scientific_name, class_name |
| `perch_score_train_sc.npz` | (10658, 12, 234) | Perch v2 on all 12 windows of all SS |
| `meta_train_sc.parquet` | 10658 rows | filename, site, hour |

## Configuration
- GPU: T4 (onnxruntime-gpu)
- Internet: True (Kaggle Dataset upload at end)
- Run time: ~60-90 min


In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import subprocess, sys, os, time, json
from pathlib import Path

# GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

# Install
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "onnxruntime-gpu", "timm>=1.0", "librosa", "soundfile", "pyarrow"])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
import librosa
import soundfile as sf
import warnings
warnings.filterwarnings("ignore")

print(f"torch={torch.__version__}, ort={ort.__version__}")
print(f"ort providers: {ort.get_available_providers()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
START_TIME = time.time()


In [ ]:
# ============================================================
# Cell 2: Load data + paths
# ============================================================
BASE = None
for p in [Path("/kaggle/input/competitions/birdclef-2026"),
          Path("/kaggle/input/birdclef-2026")]:
    if p.exists():
        BASE = p; break
assert BASE is not None
print(f"BASE: {BASE}")

TA_DIR = BASE / "train_audio"
TS_DIR = BASE / "train_soundscapes"
TAXO_PATH = BASE / "taxonomy.csv"
TRAIN_CSV = BASE / "train.csv"
SS_LABELS_PATH = BASE / "train_soundscapes_labels.csv"
SAMPLE_SUB_PATH = BASE / "sample_submission.csv"

# Label ordering
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {lbl: i for i, lbl in enumerate(PRIMARY_LABELS)}
N_CLASSES = len(PRIMARY_LABELS)
print(f"Classes: {N_CLASSES}")

# Taxonomy
taxonomy_df = pd.read_csv(TAXO_PATH)
taxonomy_df["primary_label"] = taxonomy_df["primary_label"].astype(str)
print(f"Taxonomy: {len(taxonomy_df)} species")

# Train metadata (35k focal)
train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df[train_df["primary_label"].astype(str).isin(LABEL2IDX)].reset_index(drop=True)
train_df["filename"] = train_df["filename"].astype(str)
train_df["exists"] = train_df["filename"].apply(lambda fn: (TA_DIR / fn).exists())
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print(f"train_audio: {len(train_df)} files exist")

# train_sc files
ts_files = sorted([p.name for p in TS_DIR.glob("*.ogg")])
print(f"train_sc: {len(ts_files)} files")

# Labeled SS truth
sc_labels = pd.read_csv(SS_LABELS_PATH).drop_duplicates()
if sc_labels["start"].dtype == object:
    sc_labels["start_sec"] = pd.to_timedelta(sc_labels["start"]).dt.total_seconds().astype(int)
else:
    sc_labels["start_sec"] = sc_labels["start"].astype(int)
print(f"SS labels: {len(sc_labels)} rows, {sc_labels['filename'].nunique()} files")

# Parse site/hour from filename like "BC2026_Ssite_HHMMSS"
import re
def parse_filename(fn):
    fn = str(fn)
    m = re.search(r"_S(\d+)_\d{8}_(\d{6})", fn)
    if m:
        site = "S" + m.group(1)
        hour = int(m.group(2)[:2])
        return site, hour
    return "UNK", -1

# Build Y_SC for labeled SS windows (708 windows = 59 files × 12)
labeled_files = sorted(sc_labels["filename"].unique())
print(f"Labeled files: {len(labeled_files)}")

# Each file gets 12 windows × 5s
SR = 32000
N_WINDOWS = 12
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC

ss_meta = []
for fn in labeled_files:
    site, hour = parse_filename(fn)
    for w in range(N_WINDOWS):
        ss_meta.append({
            "filename": fn,
            "site": site,
            "hour_utc": hour,
            "window_idx": w,
            "start_sec": w * WINDOW_SEC,
        })
ss_meta_df = pd.DataFrame(ss_meta)
print(f"SS meta: {len(ss_meta_df)} windows")

# Build truth (n_windows, 234)
truth_ss = np.zeros((len(ss_meta_df), N_CLASSES), dtype=np.float32)
for _, row in sc_labels.iterrows():
    fn = row["filename"]
    start = int(row["start_sec"])
    win_idx = start // WINDOW_SEC
    # Find this row in ss_meta_df
    idx_match = ss_meta_df[(ss_meta_df["filename"] == fn) & (ss_meta_df["window_idx"] == win_idx)].index
    if len(idx_match) == 0:
        continue
    win_global_idx = idx_match[0]
    for lbl in str(row["primary_label"]).split(";"):
        lbl = lbl.strip()
        if lbl in LABEL2IDX:
            truth_ss[win_global_idx, LABEL2IDX[lbl]] = 1.0

print(f"Truth SS: {truth_ss.shape}, positives={int(truth_ss.sum())}, active species={int((truth_ss.sum(axis=0)>0).sum())}")


In [ ]:
# ============================================================
# Cell 3: Perch v2 ONNX setup (GPU) - V2 FIX
# ============================================================
# Locate Perch ONNX
PERCH_PATH = None
for cand in [
    Path("/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx"),
    Path("/kaggle/input/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx"),
]:
    if cand.exists():
        PERCH_PATH = cand; break
if PERCH_PATH is None:
    for hit in Path("/kaggle/input").rglob("perch_v2*.onnx"):
        PERCH_PATH = hit; break
assert PERCH_PATH is not None, "Perch ONNX not found"
print(f"Perch: {PERCH_PATH}")

providers = [("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"]
session = ort.InferenceSession(str(PERCH_PATH), providers=providers)
input_name = session.get_inputs()[0].name
print(f"Perch input: {input_name}, providers: {session.get_providers()}")
for i, o in enumerate(session.get_outputs()):
    print(f"  output[{i}] {o.name} shape={o.shape}")

# ★ V2 FIX: Use FIRST 2D match for embedding (output[0] = pooled 1536-d)
#   Don't overwrite with spatial_embedding (4D)
EMBED_IDX = None
LABEL_IDX = None
for i, o in enumerate(session.get_outputs()):
    shape = o.shape
    if shape and EMBED_IDX is None and len(shape) == 2 and shape[-1] == 1536:
        EMBED_IDX = i
    if shape and LABEL_IDX is None and len(shape) == 2 and shape[-1] > 5000:
        LABEL_IDX = i
print(f"EMBED_IDX={EMBED_IDX}, LABEL_IDX={LABEL_IDX}")
assert EMBED_IDX is not None and LABEL_IDX is not None

def perch_infer(audio_batch, batch_size=64):
    """Run Perch inference on audio batch.
    Returns: embeddings (N, 1536), scores (N, num_perch_classes)
    """
    N = audio_batch.shape[0]
    embeddings, scores = [], []
    for i in range(0, N, batch_size):
        batch = audio_batch[i:i+batch_size].astype(np.float32)
        out = session.run(None, {input_name: batch})
        embeddings.append(out[EMBED_IDX])
        scores.append(out[LABEL_IDX])
    return np.concatenate(embeddings), np.concatenate(scores)

# Test
dummy = np.random.randn(2, SR * 5).astype(np.float32) * 0.01
emb, sc = perch_infer(dummy)
print(f"Test: emb {emb.shape}, scores {sc.shape}")
assert emb.shape[-1] == 1536, f"Embedding wrong shape: {emb.shape}"


In [ ]:
# ============================================================
# Cell 4: Perch label → BC2026 mapping (from Perch v2 TF model's assets/labels.csv)
# ============================================================
# Locate Perch v2 TF model dir (provides assets/labels.csv via model_sources)
MODEL_DIR = None
for cand in [
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"),
    Path("/kaggle/input/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"),
]:
    if cand.exists():
        MODEL_DIR = cand; break
if MODEL_DIR is None:
    # rglob fallback for labels.csv near a Perch model
    for hit in Path("/kaggle/input").rglob("labels.csv"):
        if "perch" in str(hit).lower() or "bird-vocal" in str(hit).lower():
            MODEL_DIR = hit.parent.parent
            break
assert MODEL_DIR is not None, "Perch v2 TF model dir not found"
print(f"Perch MODEL_DIR: {MODEL_DIR}")

LABELS_CSV = MODEL_DIR / "assets" / "labels.csv"
assert LABELS_CSV.exists(), f"labels.csv not found at {LABELS_CSV}"
bc_labels = pd.read_csv(LABELS_CSV)
bc_labels = bc_labels.reset_index().rename(columns={"index": "perch_class_idx"})
print(f"Perch labels.csv: {len(bc_labels)} classes")
print(f"  columns: {list(bc_labels.columns)[:8]}")
print(f"  sample: {bc_labels.head(2).to_dict('records')}")

# Determine join key — try scientific_name first, then ebird2021
join_col = None
for c in ["scientific_name", "ebird2021", "ebird_code", "code", "label"]:
    if c in bc_labels.columns and c in taxonomy_df.columns:
        join_col = c
        break
if join_col is None:
    # Try fuzzy: find a string column in bc_labels that has overlap with taxonomy
    for c in bc_labels.columns:
        if bc_labels[c].dtype == object:
            overlap = set(bc_labels[c].dropna().astype(str)) & set(taxonomy_df["scientific_name"].dropna().astype(str))
            if len(overlap) > 50:
                # rename to scientific_name to use
                bc_labels = bc_labels.rename(columns={c: "scientific_name"})
                join_col = "scientific_name"
                break
assert join_col is not None, f"No matching column found. bc_labels: {list(bc_labels.columns)}, taxonomy: {list(taxonomy_df.columns)}"
print(f"Join column: {join_col}")

NO_LABEL = len(bc_labels)
tax_with_perch = taxonomy_df[["primary_label", "scientific_name"]].merge(
    bc_labels[["perch_class_idx", join_col]],
    on=join_col if join_col in taxonomy_df.columns else "scientific_name",
    how="left",
)
tax_with_perch["perch_class_idx"] = tax_with_perch["perch_class_idx"].fillna(NO_LABEL).astype(int)

# Build BC_INDICES in PRIMARY_LABELS order
bc_idx_map = tax_with_perch.set_index("primary_label")["perch_class_idx"].to_dict()
BC_INDICES = np.array([bc_idx_map.get(lbl, NO_LABEL) for lbl in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have Perch logit")

def perch_to_bc2026(perch_scores):
    """Convert Perch (N, num_perch) logits → BC2026 (N, 234) sigmoid probs.
    Unmapped species → sigmoid(-50) ≈ 0 (NOT 0.5)."""
    bc_logits = np.full((perch_scores.shape[0], N_CLASSES), -50.0, dtype=np.float32)
    bc_logits[:, MAPPED_POS] = perch_scores[:, MAPPED_BC_IDX]
    return 1.0 / (1.0 + np.exp(-np.clip(bc_logits, -50, 50)))

test_bc = perch_to_bc2026(sc)
print(f"BC mapped test: shape={test_bc.shape}, mean={test_bc.mean():.4f}, max={test_bc.max():.4f}")
print(f"  Non-zero species (pred > 0.01): {(test_bc.max(axis=0) > 0.01).sum()}")


In [ ]:
# ============================================================
# Cell 5: Audio loader helpers
# ============================================================
def load_5s_clip(path, start_sec=0.0, duration=5.0):
    """Load 5s clip at 32kHz mono."""
    try:
        n_frames = int(SR * duration)
        wav, sr = sf.read(str(path), start=int(start_sec * SR),
                           frames=n_frames, dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        if len(wav) < n_frames:
            wav = np.pad(wav, (0, n_frames - len(wav)))
        return wav[:n_frames].astype(np.float32)
    except Exception as e:
        return np.zeros(int(SR * duration), dtype=np.float32)

def load_60s(path):
    """Load 60s mono 32kHz audio."""
    try:
        wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    except Exception:
        return np.zeros(SR * 60, dtype=np.float32)
    target = SR * 60
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    elif len(wav) > target:
        wav = wav[:target]
    return wav.astype(np.float32)

def load_focal_center_5s(path):
    """Load center 5s of focal recording (variable length)."""
    try:
        # Get full audio first
        info = sf.info(str(path))
        total_sec = info.frames / info.samplerate
        if total_sec <= 5.0:
            wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
        else:
            center = total_sec / 2.0
            start = max(0, center - 2.5)
            wav, sr = sf.read(str(path), start=int(start * info.samplerate),
                               frames=int(5.0 * info.samplerate),
                               dtype="float32", always_2d=False)
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        target = SR * 5
        if len(wav) < target:
            wav = np.pad(wav, (0, target - len(wav)))
        return wav[:target].astype(np.float32)
    except Exception:
        return np.zeros(SR * 5, dtype=np.float32)

print("Audio loaders defined")


In [ ]:
# ============================================================
# Cell 6: Phase A — Labeled SS inference (708 windows)
# ============================================================
print("\n=== Phase A: Labeled SS ===")
t0 = time.time()

# Load all 708 windows (59 files × 12 windows × 5s)
audio_ss = np.zeros((len(ss_meta_df), WINDOW_SAMPLES), dtype=np.float32)
for i, row in ss_meta_df.iterrows():
    path = TS_DIR / row["filename"]
    if not str(path).endswith(".ogg"):
        path = TS_DIR / (row["filename"] + ".ogg")
    audio_ss[i] = load_5s_clip(path, start_sec=row["start_sec"])
    if (i + 1) % 100 == 0:
        print(f"  loaded {i+1}/{len(ss_meta_df)} windows")
print(f"Loaded {len(audio_ss)} windows in {time.time()-t0:.1f}s")

# Perch GPU inference
t0 = time.time()
perch_emb_ss, perch_logits_ss = perch_infer(audio_ss, batch_size=32)
print(f"Perch SS done: emb {perch_emb_ss.shape}, logits {perch_logits_ss.shape}, time {time.time()-t0:.1f}s")

# Map to BC2026 space
perch_score_ss = perch_to_bc2026(perch_logits_ss)
print(f"BC2026 mapped: {perch_score_ss.shape}, non-zero species: {(perch_score_ss.sum(axis=0) > 0).sum()}")

# Save
OUT_DIR = Path("/kaggle/working")
np.savez_compressed(OUT_DIR / "perch_emb_ss.npz", perch_emb_ss)
np.savez_compressed(OUT_DIR / "perch_score_ss.npz", perch_score_ss)
np.savez_compressed(OUT_DIR / "truth_ss.npz", truth_ss)
ss_meta_df.to_parquet(OUT_DIR / "meta_ss.parquet")
print(f"Saved Phase A outputs to {OUT_DIR}")


In [ ]:
# ============================================================
# Cell 7: Phase B — train_audio inference (~35k files, center 5s)
# ============================================================
print("\n=== Phase B: train_audio ===")
t0 = time.time()

n_audio = len(train_df)
perch_emb_train_audio = np.zeros((n_audio, 1536), dtype=np.float32)
perch_logits_train_audio = np.zeros((n_audio, perch_logits_ss.shape[-1]), dtype=np.float32)

BATCH = 64
audio_buffer = np.zeros((BATCH, WINDOW_SAMPLES), dtype=np.float32)
for i in range(0, n_audio, BATCH):
    end = min(i + BATCH, n_audio)
    actual_batch = end - i
    # Load
    for j in range(actual_batch):
        fn = train_df.iloc[i + j]["filename"]
        audio_buffer[j] = load_focal_center_5s(TA_DIR / fn)
    # Inference
    emb, sc = perch_infer(audio_buffer[:actual_batch], batch_size=actual_batch)
    perch_emb_train_audio[i:end] = emb
    perch_logits_train_audio[i:end] = sc
    if (i + BATCH) % (BATCH * 50) == 0 or end == n_audio:
        elapsed = time.time() - t0
        eta = elapsed * (n_audio - end) / max(end, 1) / 60
        print(f"  [{end}/{n_audio}] {elapsed/60:.1f}min, ETA {eta:.1f}min")

print(f"Phase B done in {(time.time()-t0)/60:.1f}min")

# Map to BC2026
perch_score_train_audio = perch_to_bc2026(perch_logits_train_audio)
print(f"BC mapped: {perch_score_train_audio.shape}")

# Save (don't save full Perch emb to keep file size manageable for 35k)
np.savez_compressed(OUT_DIR / "perch_score_train_audio.npz", perch_score_train_audio)

# Meta
meta_audio = train_df[["filename", "primary_label", "scientific_name", "common_name", "class_name",
                         "inat_taxon_id", "latitude", "longitude", "rating", "collection"]].copy()
meta_audio.to_parquet(OUT_DIR / "meta_train_audio.parquet")
print(f"Saved Phase B outputs")


In [ ]:
# ============================================================
# Cell 8: Phase C — train_sc inference (10,658 files × 12 windows = 127k)
# ============================================================
print("\n=== Phase C: train_sc ===")
t0 = time.time()

n_files = len(ts_files)
n_total_windows = n_files * N_WINDOWS

perch_emb_train_sc = np.zeros((n_files, N_WINDOWS, 1536), dtype=np.float32)
perch_logits_train_sc = np.zeros((n_files, N_WINDOWS, perch_logits_ss.shape[-1]), dtype=np.float32)

# Process file-by-file (each file → 12 windows batch)
audio_12win = np.zeros((N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
for fi, fn in enumerate(ts_files):
    wav_60s = load_60s(TS_DIR / fn)
    for w in range(N_WINDOWS):
        audio_12win[w] = wav_60s[w * WINDOW_SAMPLES:(w + 1) * WINDOW_SAMPLES]
    emb, sc = perch_infer(audio_12win, batch_size=N_WINDOWS)
    perch_emb_train_sc[fi] = emb
    perch_logits_train_sc[fi] = sc
    if (fi + 1) % 100 == 0 or fi == n_files - 1:
        elapsed = time.time() - t0
        eta = elapsed * (n_files - fi - 1) / max(fi + 1, 1) / 60
        print(f"  [{fi+1}/{n_files}] {elapsed/60:.1f}min, ETA {eta:.1f}min")

print(f"Phase C done in {(time.time()-t0)/60:.1f}min")

# Map to BC2026
perch_score_train_sc = np.zeros((n_files, N_WINDOWS, N_CLASSES), dtype=np.float32)
for w in range(N_WINDOWS):
    perch_score_train_sc[:, w, :] = perch_to_bc2026(perch_logits_train_sc[:, w, :])
print(f"BC mapped: {perch_score_train_sc.shape}")

# Save (Perch emb too large for 10k files; skip embedding save)
np.savez_compressed(OUT_DIR / "perch_score_train_sc.npz", perch_score_train_sc)

# Meta
meta_sc = []
for fn in ts_files:
    site, hour = parse_filename(fn)
    has_truth = fn in labeled_files or fn.replace(".ogg", "") in labeled_files
    meta_sc.append({"filename": fn, "site": site, "hour_utc": hour, "has_truth": has_truth})
meta_sc_df = pd.DataFrame(meta_sc)
meta_sc_df.to_parquet(OUT_DIR / "meta_train_sc.parquet")
print(f"Saved Phase C outputs")


In [ ]:
# ============================================================
# Cell 9: Summary + output file list
# ============================================================
print("\n=== Summary ===")
print(f"Total time: {(time.time()-START_TIME)/60:.1f} min")

print("\nOutput files:")
for p in sorted(OUT_DIR.glob("*.npz")) + sorted(OUT_DIR.glob("*.parquet")):
    print(f"  {p.name}  {p.stat().st_size/1e6:.1f}MB")

# Optional: save as Kaggle Dataset (uncomment if desired)
# import tempfile, shutil
# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi(); api.authenticate()
# with tempfile.TemporaryDirectory() as td:
#     td = Path(td)
#     for p in OUT_DIR.glob("*.npz"):
#         shutil.copy(p, td / p.name)
#     for p in OUT_DIR.glob("*.parquet"):
#         shutil.copy(p, td / p.name)
#     meta = {
#         "title": "BirdCLEF2026 exp091 analysis",
#         "id": "maekeso/birdclef2026-exp091-analysis",
#         "licenses": [{"name": "CC0-1.0"}],
#     }
#     (td / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
#     api.dataset_create_version(folder=str(td), version_notes="exp091 analysis outputs",
#                                  dir_mode="zip", quiet=False)

print("\n[OK] DONE")
